# Скачивание датасета DeepFashion

## Цель
Скачать датасет DeepFashion (~6.4 GB) с Google Drive и подготовить его к использованию.

## Что делает этот ноутбук:
1. Скачивает zip-архив с датасетом
2. Показывает прогресс скачивания
3. Распаковывает архив
4. Проверяет целостность файлов
5. Показывает статистику датасета

In [5]:
# [Ячейка 2: Импорт библиотек и настройка путей]
import os
import gdown
import zipfile
from pathlib import Path
from tqdm import tqdm
import shutil

# Конфигурация
PROJECT_ROOT = Path.cwd().parent  # project/
DATA_DIR = PROJECT_ROOT / "data" / "dataset"
DOWNLOAD_DIR = PROJECT_ROOT / "data" / "downloads"

# URL для скачивания (Google Drive File ID)
FILE_ID = "1U2PljA7NE57jcSSzPs21ZurdIPXdYZtN"
ZIP_FILENAME = "deepfashion_dataset.zip"

# Создаем директории
DATA_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Project root: {PROJECT_ROOT}")
print(f"📁 Data directory: {DATA_DIR}")
print(f"📁 Download directory: {DOWNLOAD_DIR}")

📁 Project root: d:\VSCode\ЦК\aie-student-template\project
📁 Data directory: d:\VSCode\ЦК\aie-student-template\project\data\dataset
📁 Download directory: d:\VSCode\ЦК\aie-student-template\project\data\downloads


In [6]:
# [Ячейка 3: Функция для скачивания с прогресс-баром] — ИСПРАВЛЕННАЯ ВЕРСИЯ
def download_from_google_drive(file_id: str, output_path: str) -> str:
    """
    Скачивание файла с Google Drive с отображением прогресса.
    Совместимо с gdown >= 4.6
    
    Args:
        file_id: ID файла на Google Drive
        output_path: Путь для сохранения файла
    
    Returns:
        Путь к скачанному файлу
    """
    
    print(f"🔗 Starting download from Google Drive...")
    print(f"📦 File ID: {file_id}")
    print(f"💾 Saving to: {output_path}")
    print(f"⏳ This may take several minutes (file size: ~6.4 GB)...")
    print()
    
    gdown.download(id=file_id, output=output_path, quiet=False)

In [7]:
# [Ячейка 4: Скачивание датасета]
zip_path = DOWNLOAD_DIR / ZIP_FILENAME

# Проверяем, есть ли уже скачанный файл
if zip_path.exists():
    file_size_gb = os.path.getsize(zip_path) / (1024**3)
    print(f"⚠️ File already exists: {zip_path}")
    print(f"📊 Size: {file_size_gb:.2f} GB")
    
    response = input("\nDo you want to re-download? (y/n): ").strip().lower()
    if response == 'y':
        print("🗑️ Removing old file...")
        zip_path.unlink()
        downloaded_path = download_from_google_drive(FILE_ID, str(zip_path))
    else:
        downloaded_path = str(zip_path)
        print("✅ Using existing file")
else:
    downloaded_path = download_from_google_drive(FILE_ID, str(zip_path))

⚠️ File already exists: d:\VSCode\ЦК\aie-student-template\project\data\downloads\deepfashion_dataset.zip
📊 Size: 6.35 GB
✅ Using existing file


In [8]:
# [Ячейка 5: Функция для распаковки с прогресс-баром]
def extract_zip_with_progress(zip_path: str, extract_to: str) -> None:
    """
    Распаковка ZIP архива с отображением прогресса.
    
    Args:
        zip_path: Путь к ZIP файлу
        extract_to: Директория для распаковки
    """
    print(f"\n📦 Extracting archive...")
    print(f"📂 From: {zip_path}")
    print(f"📂 To: {extract_to}")
    print()
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        # Получаем список всех файлов
        file_list = zip_ref.namelist()
        total_files = len(file_list)
        
        print(f"📄 Total files in archive: {total_files}")
        
        # Создаем прогресс-бар
        with tqdm(total=total_files, desc="Extracting", unit="file") as pbar:
            # Распаковываем все файлы
            for file in file_list:
                zip_ref.extract(file, extract_to)
                pbar.update(1)
    
    print(f"\n✅ Extraction completed!")

In [9]:
# [Ячейка 6: Распаковка датасета]
# Определяем куда распаковывать
extract_path = DATA_DIR

# Проверяем, распакован ли уже датасет
if any(extract_path.iterdir()):
    print(f"⚠️ Directory {extract_path} is not empty")
    print(f"📁 Current contents: {list(extract_path.iterdir())[:5]}...")
    
    response = input("\nDo you want to extract anyway? (y/n): ").strip().lower()
    if response == 'y':
        extract_zip_with_progress(downloaded_path, str(extract_path))
    else:
        print("⏭️ Skipping extraction")
else:
    extract_zip_with_progress(downloaded_path, str(extract_path))


📦 Extracting archive...
📂 From: d:\VSCode\ЦК\aie-student-template\project\data\downloads\deepfashion_dataset.zip
📂 To: d:\VSCode\ЦК\aie-student-template\project\data\dataset

📄 Total files in archive: 44097


Extracting: 100%|██████████| 44097/44097 [01:20<00:00, 546.69file/s] 


✅ Extraction completed!


In [16]:
# [Ячейка 11: Очистка (опционально)]
# Если хотите удалить zip файл после распаковки для экономии места:

def cleanup_zip_file(zip_path: str) -> None:
    """
    Удаление ZIP архива после распаковки.
    
    Args:
        zip_path: Путь к ZIP файлу
    """
    if os.path.exists(zip_path):
        file_size_gb = os.path.getsize(zip_path) / (1024**3)
        print(f"🗑️  Deleting ZIP file: {zip_path} ({file_size_gb:.2f} GB)")
        os.remove(zip_path)
        print("✅ ZIP file deleted")
    else:
        print("ℹ️  ZIP file not found")

cleanup_zip_file(downloaded_path)

🗑️  Deleting ZIP file: d:\VSCode\ЦК\aie-student-template\project\data\downloads\deepfashion_dataset.zip (6.35 GB)
✅ ZIP file deleted


## ✅ Что сделано:

1. **Скачан датасет** с Google Drive (~6.4 GB)
2. **Распакован** в директорию `project/images/`
3. **Проанализирована** структура датасета
4. **Проверена** целостность изображений

## 📝 Следующие шаги:

1. Перейдите к `01_preprocessing.ipynb` для подготовки данных
2. Или запустите `python -m src.data.prepare` для подготовки через CLI

## 💡 Примечания:

- Если скачивание прервалось, просто запустите ячейку снова - gdown поддерживает докачку
- ZIP файл сохраняется в `project/data/downloads/` для повторного использования
- Для освобождения места можно удалить ZIP после распаковки (последняя ячейка)